# Session 1: no part is made exactly
**A. Real-world problem.** A shaft that is a little too big will not go into a hole that is
a little too small. No machine makes a part exactly 10.000 mm; every part is a bit bigger or
smaller. *Tolerance* is the wiggle we agree to allow. Today you write a part's limits, find
its tolerance, and decide whether a pin and a hole will assemble.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: limits and tolerance
A drawing gives a part two **limits**: the largest allowed size (upper limit) and the smallest
allowed size (lower limit). The **tolerance** is the difference:

    tolerance = upper limit - lower limit

A part is *in tolerance* if its measured size is between the two limits (inclusive).

In [ ]:
def tolerance(lower, upper):
    """Tolerance = upper limit - lower limit (same unit as the inputs)."""
    if upper < lower:
        raise ValueError("upper limit must not be below the lower limit")
    return round(upper - lower, 9)


def in_limits(size, lower, upper):
    """True if a measured size is inside the limits (the limits themselves are allowed)."""
    return lower - 1e-9 <= size <= upper + 1e-9

## Step 2: a pin and a hole (all sizes in mm)
Our pin is *nominal* 10 mm but is specified as at most 9.98 mm and at least `9.98 - tol`.
The hole is at least 10.00 mm and at most `10.00 + tol`. The gap between hole and pin is the
**clearance** = hole size - pin size. Negative clearance means the pin is bigger than the
hole: that is called **interference**.

In [ ]:
PIN_UPPER = 9.98
HOLE_LOWER = 10.00


def pin_limits(tol):
    return round(PIN_UPPER - tol, 9), PIN_UPPER


def hole_limits(tol):
    return HOLE_LOWER, round(HOLE_LOWER + tol, 9)


def clearance(hole_size, pin_size):
    return round(hole_size - pin_size, 9)


def kind_of_fit(c):
    if c > 1e-9:
        return "clearance"
    if c < -1e-9:
        return "interference"
    return "line to line"

## Step 3: the worst cases
The *smallest* possible clearance happens with the biggest pin in the smallest hole; the
*largest* with the smallest pin in the biggest hole:

    min clearance = hole lower limit - pin upper limit
    max clearance = hole upper limit - pin lower limit

In [ ]:
def clearance_range(tol):
    h_lo, h_hi = hole_limits(tol)
    p_lo, p_hi = pin_limits(tol)
    return clearance(h_lo, p_hi), clearance(h_hi, p_lo)


def always_assembles(tol):
    """True if every pin and hole inside their limits will go together."""
    return clearance_range(tol)[0] >= 0

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    tol = 0.05
    h_lo, h_hi = hole_limits(tol)
    p_lo, p_hi = pin_limits(tol)
    cmin, cmax = clearance_range(tol)
    print(f"hole {h_lo:.2f} to {h_hi:.2f} mm, pin {p_lo:.2f} to {p_hi:.2f} mm")
    print(f"tolerance {tolerance(p_lo, p_hi):.2f} mm; clearance {cmin:.2f} to {cmax:.2f} mm")

    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.barh(1, h_hi - h_lo, left=h_lo, height=0.5, color="tab:blue", alpha=0.6, label="hole: allowed sizes")
    ax.barh(0, p_hi - p_lo, left=p_lo, height=0.5, color="tab:orange", alpha=0.7, label="pin: allowed sizes")
    ax.axvline(10.0, color="k", ls=":", lw=1)
    ax.annotate("", xy=(h_lo, 0.5), xytext=(p_hi, 0.5), arrowprops=dict(arrowstyle="<->", color="green"))
    ax.text((h_lo + p_hi) / 2, 0.58, f"min clearance {cmin:.2f}", ha="center", fontsize=8, color="green")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["pin", "hole"])
    ax.set_xlabel("diameter (mm)")
    ax.set_title("Limits of a pin and a hole (tolerance 0.05 mm each)")
    ax.legend(fontsize=8, loc="upper center")
    ax.grid(alpha=0.3, axis="x")
    fig.tight_layout()
    fig.savefig(OUT / "limits_picture.png", dpi=120)

    tols = np.linspace(0.0, 0.2, 41)
    ranges = np.array([clearance_range(t) for t in tols])
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.fill_between(tols, ranges[:, 0], ranges[:, 1], alpha=0.3, label="possible clearances")
    ax2.plot(tols, ranges[:, 0], label="minimum clearance")
    ax2.plot(tols, ranges[:, 1], label="maximum clearance")
    ax2.set_xlabel("tolerance of each part (mm)")
    ax2.set_ylabel("clearance (mm)")
    ax2.set_title("Looser tolerance, wider spread of clearance")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "clearance_vs_tol.png", dpi=120)
    for t in (0.02, 0.05, 0.10):
        print(f"tol {t:.2f}: clearance {clearance_range(t)[0]:.2f} to {clearance_range(t)[1]:.2f} mm")

## Where this stops
Here the pin and hole limits were chosen by hand. Session 2 uses the standard ISO fit system
(such as 25 H7/g6) so that engineers everywhere choose the limits in the same way.

## Coding tasks
1. **Easy.** Change `PIN_UPPER` to 9.97 in `lesson.py` and predict the new clearance range for a tolerance of 0.05 mm before you run it.
2. **Stretch.** Write `fits_all(hole_tol, pin_tol)` that allows different tolerances for the hole and the pin, and print the clearance range for 0.03 and 0.06 mm.
3. **Explain.** Explain, in your own words, why the spread of clearance equals twice the tolerance when both parts have the same tolerance.

## Assessment
- **A. Conceptual.** Why does a drawing give limits instead of one exact size? What does a part whose size is exactly on the upper limit count as?
- **B. Numerical.** A pin is made between 9.93 and 9.98 mm and a hole between 10.00 and 10.05 mm. A 9.95 mm pin meets a 10.04 mm hole. What is the clearance, and are both parts in tolerance? Sanity check it against the clearance range 0.02 to 0.12 mm.
- **C. Graph and simulation interpretation.** In the graph of clearance against tolerance, why does the lower line stay at 0.02 mm while the upper line rises? What is the largest clearance at a tolerance of 0.10 mm?
- **D. Design and optimisation challenge.** Design: a toy needs the pin to be loose enough to always go in but not wobble more than 0.10 mm. Choose the largest whole hundredth of a millimetre for the tolerance of each part and justify it. What do you lose by choosing a smaller one?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("tolerance of 9.93 to 9.98 is 0.05", abs(L.tolerance(9.93, 9.98) - 0.05) < 1e-9))
    checks.append(("a part exactly on a limit is in tolerance", L.in_limits(9.98, 9.93, 9.98) and L.in_limits(9.93, 9.93, 9.98)))
    checks.append(("9.99 mm is out of tolerance for the pin (9.93 to 9.98)", not L.in_limits(9.99, 9.93, 9.98)))
    try:
        L.tolerance(10.0, 9.9)
        bad = False
    except ValueError:
        bad = True
    checks.append(("edge: upper below lower is rejected", bad))

    lo, hi = L.pin_limits(0.05)
    checks.append(("pin limits for tol 0.05 are 9.93 and 9.98", abs(lo - 9.93) < 1e-9 and abs(hi - 9.98) < 1e-9))
    lo, hi = L.hole_limits(0.05)
    checks.append(("hole limits for tol 0.05 are 10.00 and 10.05", abs(lo - 10.0) < 1e-9 and abs(hi - 10.05) < 1e-9))

    cmin, cmax = L.clearance_range(0.05)
    checks.append((f"worked example: clearance range is 0.02 to 0.12 mm (got {cmin:.2f} to {cmax:.2f})",
                   abs(cmin - 0.02) < 1e-9 and abs(cmax - 0.12) < 1e-9))
    checks.append(("hand calculation: tol 0.10 gives a maximum clearance of 0.22", abs(L.clearance_range(0.10)[1] - 0.22) < 1e-9))
    checks.append(("edge: tol 0 gives one clearance, 0.02 mm", L.clearance_range(0.0) == (0.02, 0.02)))
    checks.append(("clearance 0.03 is a clearance fit", L.kind_of_fit(0.03) == "clearance"))
    checks.append(("clearance -0.01 is interference", L.kind_of_fit(L.clearance(10.0, 10.01)) == "interference"))
    checks.append(("clearance 0 is line to line", L.kind_of_fit(0.0) == "line to line"))
    checks.append(("every pair assembles for these limits", L.always_assembles(0.05) and L.always_assembles(0.2)))
    # the spread of clearance is 2 * tol: loosening the tolerance widens it
    tols = (0.02, 0.05, 0.10)
    w = [L.clearance_range(t)[1] - L.clearance_range(t)[0] for t in tols]
    checks.append(("spread of clearance equals 2 x tolerance", all(abs(a - 2 * t) < 1e-9 for a, t in zip(w, tols))))
    # a pin of 9.99 mm is out of spec, yet it would still fit a 10.00 mm hole: tolerance is about the spec
    checks.append(("a 9.99 pin still fits a 10.00 hole (0.01 mm clearance) but is rejected by the drawing",
                   abs(L.clearance(10.0, 9.99) - 0.01) < 1e-9 and not L.in_limits(9.99, *L.pin_limits(0.05))))
    # numbers used in the assessment questions
    checks.append(("tol 0.04 gives clearance 0.02 to 0.10; tol 0.02 gives 0.02 to 0.06",
                   L.clearance_range(0.04) == (0.02, 0.10) and L.clearance_range(0.02) == (0.02, 0.06)))
    checks.append(("a 9.95 pin in a 10.04 hole has 0.09 mm clearance, and both are in tolerance for tol 0.05",
                   abs(L.clearance(10.04, 9.95) - 0.09) < 1e-9 and L.in_limits(9.95, *L.pin_limits(0.05)) and L.in_limits(10.04, *L.hole_limits(0.05))))
    checks.append(("tol 0.10 gives a minimum clearance of 0.02 mm and tol 0.01 a maximum of 0.04", L.clearance_range(0.10)[0] == 0.02 and L.clearance_range(0.01)[1] == 0.04))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
